# Tree-Based Regression Models for NYC Taxi Trip Duration Prediction

---

# Project

**NYC Taxi Trip Duration Prediction**

This notebook trains and evaluates the required tree-based regression models for predicting NYC Yellow Taxi trip duration.

Tree-based algorithms can capture complex non-linear relationships and feature interactions without requiring feature scaling. They often outperform linear models on structured tabular datasets.

The feature-engineered dataset created in the previous phase will be used to train and compare the following tree-based models.

---

# Objectives

This notebook will:

- Load the feature-engineered dataset
- Prepare features and target variable
- Perform a time-based train/validation/test split
- Train the required tree-based regression models
- Evaluate every model using identical metrics
- Compare model performance
- Save trained models
- Save evaluation results

---

# Models

Required models:

1. Decision Tree Regressor
2. Random Forest Regressor
3. Gradient Boosting Regressor

---

# Evaluation Metrics

Each model will be evaluated using:

- Mean Absolute Error (MAE)
- Root Mean Squared Error (RMSE)
- Median Absolute Error
- Coefficient of Determination (R²)
- Training Time
- Prediction Time

---

# Output

Generated artifacts:

```
models/
├── decision_tree.joblib
├── random_forest.joblib
├── gradient_boosting.joblib
└── best_tree_model.joblib

reports/
└── tree_results.csv
```

---

# Workflow

```
Feature Dataset
        │
        ▼
Load Dataset
        │
        ▼
Train / Validation / Test Split
        │
        ▼
Train Tree Models
        │
        ▼
Evaluate Models
        │
        ▼
Compare Results
        │
        ▼
Save Models & Reports
```

## Project Setup

In [3]:
from pathlib import Path
import sys


PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


sys.path.insert(
    0,
    str(PROJECT_ROOT)
)


print(PROJECT_ROOT)

c:\Users\abuba\Downloads\ML-Projects\Day-5


## Import 


In [4]:
import pandas as pd
import numpy as np


from src.data.load_data import (
    load_featured_data,
)


from src.models.train import (
    train_dummy_baseline,
    train_decision_tree,
    train_random_forest,
    train_gradient_boosting,
)


from src.models.evaluate import (
    evaluate_model,
)


from src.models.utils import (
    save_model,
    save_results,
)

# Load Data

In [5]:
taxi_df = load_featured_data()


print(
    "Dataset Shape:",
    taxi_df.shape
)


taxi_df.head()

Dataset Shape: (18378485, 32)


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,congestion_surcharge,...,hour_sin,hour_cos,day_sin,day_cos,same_borough,same_location,is_airport_trip,distance_squared,log_trip_distance,distance_per_passenger
0,2,2023-01-01 00:32:10,2023-01-01 00:40:36,1,0.97,1.0,N,161,141,2.5,...,0.0,1.0,-0.781832,0.62349,1,0,0,0.9409,0.678034,0.97
1,2,2023-01-01 00:55:08,2023-01-01 01:01:27,1,1.10,1.0,N,43,237,2.5,...,0.0,1.0,-0.781832,0.62349,1,0,0,1.2100,0.741937,1.10
2,2,2023-01-01 00:25:04,2023-01-01 00:37:49,1,2.51,1.0,N,48,238,2.5,...,0.0,1.0,-0.781832,0.62349,1,0,0,6.3001,1.255616,2.51
3,2,2023-01-01 00:10:29,2023-01-01 00:21:19,1,1.43,1.0,N,107,79,2.5,...,0.0,1.0,-0.781832,0.62349,1,0,0,2.0449,0.887891,1.43
4,2,2023-01-01 00:50:34,2023-01-01 01:02:52,1,1.84,1.0,N,161,137,2.5,...,0.0,1.0,-0.781832,0.62349,1,0,0,3.3856,1.043804,1.84


In [6]:
taxi_df.columns.tolist()

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'congestion_surcharge',
 'airport_fee',
 'trip_duration_minutes',
 'log_trip_duration',
 'pickup_year',
 'pickup_month',
 'pickup_day',
 'pickup_hour',
 'pickup_dayofweek',
 'pickup_week',
 'pickup_weekend',
 'pickup_quarter',
 'rush_hour',
 'hour_sin',
 'hour_cos',
 'day_sin',
 'day_cos',
 'same_borough',
 'same_location',
 'is_airport_trip',
 'distance_squared',
 'log_trip_distance',
 'distance_per_passenger']

In [7]:
taxi_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 18378485 entries, 0 to 18378484
Data columns (total 32 columns):
 #   Column                  Dtype         
---  ------                  -----         
 0   VendorID                int64         
 1   tpep_pickup_datetime    datetime64[us]
 2   tpep_dropoff_datetime   datetime64[us]
 3   passenger_count         int8          
 4   trip_distance           float32       
 5   RatecodeID              float64       
 6   store_and_fwd_flag      str           
 7   PULocationID            int64         
 8   DOLocationID            int64         
 9   congestion_surcharge    float64       
 10  airport_fee             float64       
 11  trip_duration_minutes   float32       
 12  log_trip_duration       float64       
 13  pickup_year             int16         
 14  pickup_month            int8          
 15  pickup_day              int8          
 16  pickup_hour             int8          
 17  pickup_dayofweek        int8          
 18  pickup_week

## Features and Target

In [8]:
TARGET = "log_trip_duration"

X = taxi_df.drop(
    columns=[TARGET]
)

y = taxi_df[TARGET]

In [9]:
# =============================================================================
# Prepare Features and Target
# =============================================================================

TARGET = "log_trip_duration"

# Columns not used for training

DROP_COLUMNS = [
    "log_trip_duration",          # Target
    "trip_duration_minutes",      # Original target (data leakage)
    "tpep_pickup_datetime",       # Raw datetime
    "tpep_dropoff_datetime",      # Raw datetime / leakage risk
]


X = taxi_df.drop(
    columns=DROP_COLUMNS
)

y = taxi_df[TARGET]


print("=" * 70)
print("Dataset Prepared for Tree Models")
print("=" * 70)

print(f"Feature Matrix Shape : {X.shape}")
print(f"Target Shape         : {y.shape}")
print(f"Number of Features   : {X.shape[1]}")

X.head()

Dataset Prepared for Tree Models
Feature Matrix Shape : (18378485, 28)
Target Shape         : (18378485,)
Number of Features   : 28


,VendorID,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,congestion_surcharge,airport_fee,pickup_year,...,hour_sin,hour_cos,day_sin,day_cos,same_borough,same_location,is_airport_trip,distance_squared,log_trip_distance,distance_per_passenger
0,2,1,0.97,1.0,N,161,141,2.5,0.0,2023,...,0.0,1.0,-0.781832,0.62349,1,0,0,0.9409,0.678034,0.97
1,2,1,1.10,1.0,N,43,237,2.5,0.0,2023,...,0.0,1.0,-0.781832,0.62349,1,0,0,1.2100,0.741937,1.10
2,2,1,2.51,1.0,N,48,238,2.5,0.0,2023,...,0.0,1.0,-0.781832,0.62349,1,0,0,6.3001,1.255616,2.51
3,2,1,1.43,1.0,N,107,79,2.5,0.0,2023,...,0.0,1.0,-0.781832,0.62349,1,0,0,2.0449,0.887891,1.43
4,2,1,1.84,1.0,N,161,137,2.5,0.0,2023,...,0.0,1.0,-0.781832,0.62349,1,0,0,3.3856,1.043804,1.84


## Train/Test Split

In [10]:
from sklearn.model_selection import train_test_split


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


print(X_train.shape)
print(X_test.shape)

(14702788, 28)
(3675697, 28)


In [11]:
# =============================================================================
# Preprocessing Pipeline
# =============================================================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder


# Identify feature types

numeric_features = X_train.select_dtypes(
    include=[
        "int8",
        "int16",
        "int64",
        "float32",
        "float64"
    ]
).columns.tolist()


categorical_features = X_train.select_dtypes(
    include=[
        "object",
        "string"
    ]
).columns.tolist()


print("=" * 70)
print("Feature Types")
print("=" * 70)

print(f"Numerical Features     : {len(numeric_features)}")
print(f"Categorical Features   : {len(categorical_features)}")

print("\nCategorical Columns:")
print(categorical_features)



# Create preprocessing pipeline

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            "passthrough",
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        ),
    ]
)



# Fit on training data and transform

X_train_processed = preprocessor.fit_transform(
    X_train
)


X_test_processed = preprocessor.transform(
    X_test
)



print("=" * 70)
print("Preprocessing Completed")
print("=" * 70)

print(f"X_train_processed Shape : {X_train_processed.shape}")
print(f"X_test_processed Shape  : {X_test_processed.shape}")

Feature Types
Numerical Features     : 27
Categorical Features   : 1

Categorical Columns:
['store_and_fwd_flag']
Preprocessing Completed
X_train_processed Shape : (14702788, 29)
X_test_processed Shape  : (3675697, 29)


In [12]:
# =============================================================================
# Prepare Training Sample
# =============================================================================

SAMPLE_SIZE = 2000000


X_train_model = X_train_processed[:SAMPLE_SIZE]

y_train_model = y_train.iloc[:SAMPLE_SIZE]


print("=" * 70)
print("Training Sample Prepared")
print("=" * 70)


print("X_train_model Shape:", X_train_model.shape)
print("y_train_model Shape:", y_train_model.shape)

Training Sample Prepared
X_train_model Shape: (2000000, 29)
y_train_model Shape: (2000000,)


In [13]:
# =============================================================================
# Model Storage
# =============================================================================

models = {}
training_times = {}


print("Model storage initialized")

Model storage initialized


## Train Dummy Baseline

In [14]:
models["Decision Tree"], training_times["Decision Tree"] = train_decision_tree(
    X_train_model,
    y_train_model
)

## 
Train Decision Tree

In [15]:
models["Decision Tree"], training_times["Decision Tree"] = train_decision_tree(
    X_train_processed,
    y_train
)

## Train Random Forest

In [17]:
models["Random Forest"], training_times["Random Forest"] = train_random_forest(
    X_train_model,
    y_train_model,
    n_estimators=50
)

## Train Gradient Boosting

In [18]:
# =============================================================================
# Gradient Boosting Regressor
# =============================================================================

GB_SAMPLE_SIZE = 500000


X_train_gb = X_train_processed[:GB_SAMPLE_SIZE]

y_train_gb = y_train.iloc[:GB_SAMPLE_SIZE]


models["Gradient Boosting"], training_times["Gradient Boosting"] = train_gradient_boosting(
    X_train_gb,
    y_train_gb
)


print(
    f"Gradient Boosting completed in "
    f"{training_times['Gradient Boosting']:.3f} seconds"
)

Gradient Boosting completed in 115.825 seconds


In [19]:
# =============================================================================
# Save Trained Models
# =============================================================================

from pathlib import Path


PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


MODELS_DIR = PROJECT_ROOT / "models"


save_model(
    models["Decision Tree"],
    "decision_tree",
    MODELS_DIR
)


save_model(
    models["Random Forest"],
    "random_forest",
    MODELS_DIR
)


save_model(
    models["Gradient Boosting"],
    "gradient_boosting",
    MODELS_DIR
)


# Save preprocessing pipeline

save_model(
    preprocessor,
    "preprocessor",
    MODELS_DIR
)


print("=" * 70)
print("All Models Saved Successfully")
print("=" * 70)

All Models Saved Successfully
